Bronze Delta (raw_json)
   │  read + filters (interval/source/symbol/date)
   ▼
Parse JSON (from_json) → Extract klines
   ▼
Explode klines → 1 row per 1m bar
   ▼
Transform → bar_start_ts, bar_end_ts, OHLCV, p_date
   ▼
DQ checks + Dedup
   ▼
Silver Delta (partition p_date)
   ▼
MERGE (upsert)
   ▼
Validation queries


# Cell 1. Dependencies & Imports

In [0]:
import json
from datetime import datetime, timezone
from typing import Optional

from pyspark.sql import functions as F
from pyspark.sql import types as T

# Cell 2. Widgets (Parameters)

In [0]:
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("bronze_schema", "bronze_ingestion")
dbutils.widgets.text("silver_schema", "silver_ingestion")

dbutils.widgets.text("bronze_table", "brz_crypto_ohlc_1m")
dbutils.widgets.text("silver_table", "slv_crypto_ohlc_1m")

dbutils.widgets.text("interval", "1m")   # Fixed to 1m for this pipeline

dbutils.widgets.text("source", "coinbase")  # Optional filter
dbutils.widgets.text("symbols", "")         # Optional filter
dbutils.widgets.text("start_date", "")      # Optional filter
dbutils.widgets.text("end_date", "")        # Optional filter

CATALOG = dbutils.widgets.get("catalog").strip()
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema").strip()
SILVER_SCHEMA = dbutils.widgets.get("silver_schema").strip()

BRONZE_TABLE = dbutils.widgets.get("bronze_table").strip()
SILVER_TABLE = dbutils.widgets.get("silver_table").strip()
INTERVAL = dbutils.widgets.get("interval").strip()

SOURCE = dbutils.widgets.get("source").strip()
SYMBOLS_RAW = dbutils.widgets.get("symbols").strip()
SYMBOLS = [s.strip().upper() for s in SYMBOLS_RAW.split(",") if s.strip()] if SYMBOLS_RAW else []

START_DATE = dbutils.widgets.get("start_date").strip()
END_DATE = dbutils.widgets.get("end_date").strip()

BRONZE_TBL_FULL = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"
SILVER_TBL_FULL = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"

print(f"[INFO] Bronze: {BRONZE_TBL_FULL}")
print(f"[INFO] Silver: {SILVER_TBL_FULL}")
print(f"[INFO] Interval={INTERVAL}")

# Cell 3. Ensure Silver Schema Exists

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

# Cell 4. Read Bronze Data

In [0]:
bronze = spark.read.table(BRONZE_TBL_FULL)

# 1. Filter by Interval (Safety verification)
bronze = bronze.filter(F.col("interval") == INTERVAL)

# 2. Optional Filters
if SOURCE:
    bronze = bronze.where(F.col("source") == F.lit(SOURCE))

if SYMBOLS:
    bronze = bronze.where(F.col("symbol").isin(SYMBOLS))

# 3. Event Time Filter (Optimization for incremental runs)
if START_DATE:
    bronze = bronze.where(F.col("event_time") >= F.to_timestamp(F.lit(f"{START_DATE} 00:00:00")))
if END_DATE:
    bronze = bronze.where(F.col("event_time") <= F.to_timestamp(F.lit(f"{END_DATE} 23:59:59")))

# Cell5. Select & Transform Columns

In [0]:
silver_df = (bronze
  # 1. Timestamp: event_ts is Long (unix seconds), cast to Timestamp
  .withColumn("bar_start_ts", F.col("event_ts").cast("timestamp"))
  
  # 2. End Time (Start + 1 Minute)
  .withColumn("bar_end_ts", F.col("event_ts").cast("timestamp") + F.expr("INTERVAL 1 MINUTE"))
  
  # 3. Cast Numeric Columns (just in case, though 41 writes Double)
  .withColumn("open",   F.col("open").cast("double"))
  .withColumn("high",   F.col("high").cast("double"))
  .withColumn("low",    F.col("low").cast("double"))
  .withColumn("close",  F.col("close").cast("double"))
  .withColumn("volume", F.col("volume").cast("double"))
  
  # 4. Ingest Timestamp
  .withColumn("ingest_ts", F.col("ingest_ts").cast("timestamp"))
  
  # 5. Partition Date
  .withColumn("p_date", F.to_date(F.col("event_ts").cast("timestamp")))
  
  .select(
      "source", "symbol", "bar_start_ts", "bar_end_ts",
      "open", "high", "low", "close", "volume",
      "ingest_ts", "p_date"
  )
)

print(f"Transformed rows count: {silver_df.count()}")

print(f"Parsed rows count: {silver_df.count()}")

# Cell 6. Data Quality & Deduplication

In [0]:
# 1. Basic Quality Checks
silver_df = (silver_df
  .where(F.col("bar_start_ts").isNotNull())
  .where(F.col("open") > 0)
  .where(F.col("high") >= F.greatest(F.col("open"), F.col("close")))
  .where(F.col("low") <= F.least(F.col("open"), F.col("close")))
)

# 2. Intra-batch Deduplication
# Keep the latest ingestion if duplicates exist for the same bar
silver_df = (silver_df
             .dropDuplicates(["source", "symbol", "bar_start_ts"]))

print(f"Cleaned rows count: {silver_df.count()}")

# COMMAND ----------

# Cell 7. Create Silver Table (DDL)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TBL_FULL} (
  source        STRING,
  symbol        STRING,
  interval      STRING, 
  bar_start_ts  TIMESTAMP,
  bar_end_ts    TIMESTAMP,
  open          DOUBLE,
  high          DOUBLE,
  low           DOUBLE,
  close         DOUBLE,
  volume        DOUBLE,
  ingest_ts     TIMESTAMP,
  p_date        DATE
)
USING DELTA
PARTITIONED BY (p_date)
COMMENT 'Silver Layer: Cleaned 1-minute OHLC data from API-S3-Databricks pipeline'
""")

# Cell 8. MERGE (Upsert) into Silver Table

In [0]:
# Create Temp View for Merge
stg_view = "stg_slv_crypto_ohlc_1m"
silver_df = silver_df.withColumn("interval", F.lit(INTERVAL)) # Ensure interval column exists
silver_df.createOrReplaceTempView(stg_view)

spark.sql(f"""
MERGE INTO {SILVER_TBL_FULL} AS tgt
USING {stg_view} AS src
ON  tgt.source = src.source
AND tgt.symbol = src.symbol
AND tgt.interval = src.interval
AND tgt.bar_start_ts = src.bar_start_ts
WHEN MATCHED THEN UPDATE SET
  tgt.bar_end_ts     = src.bar_end_ts,
  tgt.open           = src.open,
  tgt.high           = src.high,
  tgt.low            = src.low,
  tgt.close          = src.close,
  tgt.volume         = src.volume,
  tgt.ingest_ts      = src.ingest_ts,
  tgt.p_date         = src.p_date
WHEN NOT MATCHED THEN INSERT *
""")

print("[SUCCESS] Merge completed.")

# Cell 9. Verification

In [0]:
# Check recent data
display(spark.sql(f"SELECT * FROM {SILVER_TBL_FULL} ORDER BY bar_start_ts DESC LIMIT 20"))

# Check distinct symbols
display(spark.sql(f"SELECT source, symbol, count(1) as cnt FROM {SILVER_TBL_FULL} GROUP BY source, symbol"))
